# Rhea FinGraph — XGBoost Training on Kaggle T4

Trains both model variants on the free Tesla T4 GPU (zero MacBook heat):
1. **online** — cold-start-safe features; this serves the live API
2. **full** — history-aware benchmark variant

**Before running:** Session options → Accelerator → **GPU T4 x2** (falls back to CPU automatically if left off), and Input must include your `rhea-fingraph-ibm-splits` dataset.
**After running:** File → Save Version → **Save & Run All (Commit)**, wait for *Save complete*, then download from the Output page.

In [ ]:
%pip install -q -U xgboost polars
%pip install -q --force-reinstall --no-deps git+https://github.com/aditisahu1234/Rhea-FinGraph.git

In [ ]:
import glob
import torch

paths = {p.split("/")[-1]: p for p in glob.glob("/kaggle/input/**/*.parquet", recursive=True)}
print("Found:", sorted(paths))
assert {"train.parquet", "validation.parquet", "test.parquet"} <= set(paths), paths
TRAIN, VAL, TEST = paths["train.parquet"], paths["validation.parquet"], paths["test.parquet"]

# Fall back to CPU instead of crashing when the GPU was left disabled.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU enabled - training on CPU (slower but works).")
    print("To use the T4: Session options -> Accelerator -> GPU T4 x2, then re-run.")

## 1) Train the ONLINE (serving) model

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "fingraph_sentinel.train_baseline",
        "--backend", "xgboost", "--device", DEVICE, "--feature-set", "online",
        "--train", TRAIN, "--val", VAL, "--test", TEST,
        "--out", "/kaggle/working/baseline-online-xgb",
    ],
    check=True,
)

## 2) Train the FULL (benchmark) model

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "fingraph_sentinel.train_baseline",
        "--backend", "xgboost", "--device", DEVICE, "--feature-set", "full",
        "--train", TRAIN, "--val", VAL, "--test", TEST,
        "--out", "/kaggle/working/baseline-full-xgb",
    ],
    check=True,
)

In [ ]:
!cd /kaggle/working && zip -qr rhea_xgb_artifacts.zip baseline-online-xgb baseline-full-xgb && ls -lh rhea_xgb_artifacts.zip
print("\nDone. Save Version -> Save & Run All, then download rhea_xgb_artifacts.zip from the Output page.")